# 09. From deposited energy to charge on the pixels

**The question.** A charged particle that crosses a silicon sensor leaves energy along its path.
How does that energy become charge on the pixels, and what does the charge of one crossing look
like?

This notebook follows one simulated shower through the first stages of the chain: the energy
Geant4 records in every step in the silicon, the groups and jobs that `deposit_jobs.py` hands to
PIXELAV, and the charge PIXELAV collects on each pixel. It ends with the charge image of one
sensor plane, which is where notebook 10 starts.

**Where this notebook sits in the chain**

```
Geant4 shower: the energy deposited in every step in the silicon   steps_b0_evt2.npz  (1 shower)
  -> deposit_jobs.py: groups of steps, cut into jobs for PIXELAV    jobs_b0.npz        (20 showers)
  -> PIXELAV: the charge each group leaves on each pixel            charges_b0.npz     (20 showers)
  -> this notebook: from the steps to the charge image of one plane
  -> notebook 10: one plane, its reticles, rising density
  -> notebook 11: the count of each reticle estimated from its hit map
  -> notebook 12: the counts summed over a shower
```

**What you need beforehand**

- An electromagnetic shower: a high-energy photon converts into an electron and a positron in
  the tungsten, these radiate new photons, and so on, until the energy is shared among many
  low-energy particles. Notebook 01 shows this for the whole calorimeter.
- The detector: 30 layers of silicon sensors behind tungsten plates, each sensor 100 um thick
  and cut into pixels of 50 um x 25 um. Notebook 10 describes it in one paragraph.
- Three input files, in `models/reticle_study/` or in the directory named by the environment
  variable `CALOMAPS_RETICLE_STUDY`:
  - `steps_b0_evt2.npz`: every Geant4 step in the silicon of shower 2 of batch 0 (a 50 GeV
    photon). It is cut from the batch's step file by `analysis/pixelav/step_subset.py`.
  - `jobs_b0.npz`: the groups and jobs of the batch's 20 showers, written by
    `analysis/pixelav/deposit_jobs.py`.
  - `charges_b0.npz`: the charge of every group on every pixel for the same 20 showers, written
    by `analysis/pixelav/charge_maps.py` from PIXELAV's outputs.
- The docstrings of `deposit_jobs.py` and `charge_maps.py` define the words used below. This
  notebook runs the first again on shower 2. The second needs PIXELAV's output files, which are
  not among the three files.
- Making the files again: the step file comes from `sim/make_pixelav_inputs.sh` with
  `CALOMAPS_PIXELAV_ROUTE=deposits`; the charges need PIXELAV and a sensor field map that is not
  in this repository. The commands are in sections 2, 4 and 5.3 of `docs/pixelav_handoff.md` and
  sections 1, 2 and 9 of `docs/reticle_mip_counting.md`.

**Words used below**

- *step*: one Geant4 step of one particle inside the silicon, a short piece of its path. Here a
  charged particle's steps are at most 5 um long.
- *deposit*: the energy a step leaves in the silicon, in keV or eV.
- *electron-hole pair*: the free charge a deposit makes in the silicon, on average one pair per
  3.68 eV. The electrons drift to the pixels.
- *pixel charge*: the charge collected on a pixel, counted in electrons (e); 1 ke = 1000 e.
- *sensor plane*: one layer of one module (one face of the barrel) in one shower. On a plane,
  `u` runs across the module, `v` along the barrel axis, and the depth `w` along the outward
  normal.
- *delta ray*: an electron knocked out of an atom by a passing charged particle, fast enough to
  travel on its own.
- *group*: one charged particle in one sensor plane, together with the delta rays it made in
  that slab of silicon (section 3).
- *tile* and *window*: every sensor plane is covered by tiles of 17 x 9 pixels. A window is a
  tile plus a guard band of 2 pixels on every side, 21 x 13 pixels: the area on which PIXELAV
  computes the charge (section 4).
- *job*: the part of one group that lies in one tile, handed to PIXELAV as a list of short
  straight pieces with their energies (section 4).

## 1. The steps of one shower

First the configuration and the three files. The checks at the end of the cell make sure the
files belong together.

In [ ]:
import os, sys, time, io, tempfile, contextlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
from scipy.optimize import curve_fit
from scipy.stats import norm

T_START = time.time()
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA = os.environ.get('CALOMAPS_RETICLE_STUDY', os.path.join(REPO, 'models', 'reticle_study'))
sys.path.insert(0, os.path.join(REPO, 'analysis'))
sys.path.insert(0, os.path.join(REPO, 'analysis', 'pixelav'))
import sensor_crossings as sc    # the sensor-local frame (analysis/sensor_crossings.py)
import deposit_jobs as dj        # steps -> groups -> jobs (analysis/pixelav/deposit_jobs.py)
import charge_maps as cm         # PIXELAV outputs -> charge per group and pixel (analysis/pixelav/charge_maps.py)


def need(name):
    # the path of an input file, or a clear error naming it
    path = os.path.join(DATA, name)
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{path} is missing. steps_b0_evt2.npz is cut by analysis/pixelav/step_subset.py from a step file of "
            "sim/make_pixelav_inputs.sh (CALOMAPS_PIXELAV_ROUTE=deposits); jobs_b0.npz is written by "
            "analysis/pixelav/deposit_jobs.py and charges_b0.npz by analysis/pixelav/charge_maps.py; see "
            "docs/pixelav_handoff.md sections 2, 4 and 5.3. Put them in models/reticle_study/ or point "
            "CALOMAPS_RETICLE_STUDY at their directory.")
    return path


EVENT = 2              # the shower followed here (the display shower of notebook 10)
THRESHOLD = 1000.0     # electrons: the pixel threshold of notebooks 10 and 11; a crossing counts above it
NORMAL = 1.03          # a normal crossing: a path in the slab at most 3% longer than the thickness
print(f"repository: {REPO}\ndata: {DATA}")
print(f"numpy {np.__version__}; threshold {THRESHOLD:g} e; a normal crossing: a path of at most {NORMAL} x the "
      f"thickness ({100 * (NORMAL - 1):.0f}% longer)")

In [ ]:
with np.load(need('steps_b0_evt2.npz'), allow_pickle=False) as d:
    ST = {k: d[k] for k in d.files}
with np.load(need('jobs_b0.npz'), allow_pickle=False) as d:
    JB = {k: d[k] for k in d.files}
with np.load(need('charges_b0.npz'), allow_pickle=False) as d:
    CH = {k: d[k] for k in d.files}

PITCH_V, PITCH_U = float(JB['pitch_x_um']), float(JB['pitch_y_um'])   # um: v along the barrel axis, u across
THICK = float(JB['thick_um'])                    # um of silicon
EV_PER_PAIR = float(JB['ev_per_pair'])           # eV per electron-hole pair
PIECE = float(JB['piece_um'])                    # um: the longest piece of a job
STEP_LIMIT = float(ST['geo_si_step_mm']) * 1e3   # um: the longest Geant4 step in the silicon
TILE_X, TILE_Y, GUARD = int(JB['tile_x']), int(JB['tile_y']), int(JB['guard'])
WIN_X, WIN_Y = int(JB['window_x']), int(JB['window_y'])
CENTRE_X, CENTRE_Y = int(JB['centre_x']), int(JB['centre_y'])
N_LAYERS = len(sc.si_layer_centers())
N_SLABS = sc.N_FACES * N_LAYERS                  # (module, layer) pairs
FIRST_PART = int(ST['subset_first_particle'])    # the batch's particle row that is row 0 here

assert str(ST['subset_source']) == str(JB['source_npz']), "the step file is not cut from the batch's step file"
assert int(ST['subset_event']) == EVENT and int(ST['num_events']) == 1, "the step file is not shower 2 alone"
assert all(np.array_equal(JB[k], CH[k]) for k in JB if k.startswith('group_')), "the jobs and charges files disagree"
assert np.array_equal(JB['job_id'], CH['job_id']), "the jobs and charges files hold other jobs"
assert EV_PER_PAIR == dj.EV_PER_PAIR and abs(float(ST['geo_si_thick_mm']) * 1e3 - THICK) < 1e-9
print(f"batch 0: {int(JB['num_events'])} showers of {float(JB['sim_gun_energy_GeV']):g} GeV {JB['sim_gun_particle']}, "
      f"simulation seed {JB['sim_seed']}, {JB['sim_geant4_version']}")
print(f"silicon {THICK:g} um thick in {N_LAYERS} layers, step limit {STEP_LIMIT:g} um, production cut "
      f"{float(ST['geo_si_cut_mm']) * 1e3:g} um; pixels {PITCH_V:g} x {PITCH_U:g} um (v x u)")
print(f"{EV_PER_PAIR} eV per pair; window {WIN_X} x {WIN_Y} pixels = tile {TILE_X} x {TILE_Y} + a guard band of "
      f"{GUARD} pixels, window pixel ({CENTRE_X}, {CENTRE_Y}) centred on (0, 0); pieces of at most {PIECE:g} um; "
      f"at most {int(JB['max_pairs']):,} pairs per job")
print(f"check: the step file is shower {EVENT} of the step file the jobs were built from (its particle row 0 is "
      f"batch row {FIRST_PART:,}), and the jobs and charges files hold the same groups and jobs")

The step file has one row per particle and one row per step:

| array | one entry per | meaning |
|---|---|---|
| `pdg`, `charge`, `mass` | particle | the type (11 electron, -11 positron, 22 photon), charge and mass [GeV] |
| `px`, `py`, `pz`, `E` | particle | momentum and energy where it was made [GeV] |
| `vsx`, `vsy`, `vsz` and `vex`, `vey`, `vez` | particle | where it was made and where it ended [mm] |
| `parent`, `parent_pdg`, `parent_charged` | particle | the row of its parent in this file (-1 when the parent is not in it), the parent's type, and whether the parent is charged |
| `thx`, `thy`, `thz` | step | the midpoint of the step [mm] |
| `tpx`, `tpy`, `tpz` | step | the mean momentum over the step [GeV] |
| `tpath`, `tedep`, `ttime` | step | the step length [mm], its deposit [GeV], its time [ns] |
| `tmodule`, `tlayer` | step | the module (barrel face) and layer of its sensor, from the cellID |
| `tmc` | step | the row of the particle that made the step |

Two arrays link the rows: `tmc` points from a step to its particle, and `parent` from a particle
to its parent. The file keeps only particles that left at least one step in the silicon, so a
parent that never touched the silicon is not in it: then `parent` is -1, while `parent_pdg` still
names its type.

In [ ]:
n_part, n_step = len(ST['pdg']), len(ST['tedep'])
tmc, parent = ST['tmc'].astype(np.int64), ST['parent'].astype(np.int64)
edep_eV = ST['tedep'].astype(np.float64) * 1e9           # deposit of every step [eV]
path_um = ST['tpath'].astype(np.float64) * 1e3           # length of every step [um]
E_TOTAL_KEV = edep_eV.sum() / 1e3
steps_per_part = np.bincount(tmc, minlength=n_part)
known = parent >= 0
assert tmc.min() >= 0 and tmc.max() < n_part, "a step points outside the particle rows"
assert np.all(steps_per_part > 0), "a particle in the file has no step"
assert np.array_equal(ST['pdg'][parent[known]], ST['parent_pdg'][known]), "parent and parent_pdg disagree"
print(f"shower {EVENT}: {n_part:,} particles, {n_step:,} steps, {E_TOTAL_KEV:,.3f} keV deposited in the silicon")
print(f"check: every tmc is a particle row, every particle has a step (median {np.median(steps_per_part):.0f}, "
      f"most {steps_per_part.max():,}), and parent agrees with parent_pdg")

PDG_NAME = {11: 'e-', -11: 'e+', 22: 'photon', 2112: 'neutron', 2212: 'proton'}


def pdg_name(p):
    return PDG_NAME.get(int(p), 'nucleus' if abs(int(p)) > 1000000000 else str(int(p)))


lost_type, lost_n = np.unique(ST['parent_pdg'][~known], return_counts=True)
print(f"{int(known.sum()):,} particles have their parent in the file; {int((~known).sum()):,} have parent -1, with "
      "parents of type " + ", ".join(f"{pdg_name(t)} {n:,}" for t, n in zip(lost_type, lost_n)))
step_pdg = ST['pdg'][tmc]
print(f"\n{'type':8s} {'particles':>9s} {'steps':>8s} {'deposit [keV]':>14s} {'share':>7s}")
for p in sorted(np.unique(ST['pdg']), key=lambda p: -edep_eV[step_pdg == p].sum()):
    e = edep_eV[step_pdg == p].sum() / 1e3
    print(f"{pdg_name(p):8s} {int(np.sum(ST['pdg'] == p)):9,d} {int(np.sum(step_pdg == p)):8,d} {e:14,.1f} "
          f"{100 * e / E_TOTAL_KEV:6.2f}%")

Electrons carry most of the deposit, positrons most of the rest; silicon nuclei and photons leave
a negligible share.

**The sensor frame.** Each step carries the module and layer of its sensor (`tmodule`,
`tlayer`). `sensor_crossings.tracker_hit_frame` turns them into the direction of that module's
outward normal and the depth `w` of the step along it, and `sensor_crossings.to_local` gives
`u` and `v`. We do not decode the cellID by hand. From `w` we take the depth inside the slab,
`z`, from 0 on the inner face to 100 um on the outer one (the convention of `deposit_jobs.py`).

In [ ]:
phin, w_mm, layer = sc.tracker_hit_frame(ST)            # face normal [rad], depth w [mm], layer
u_mm, v_mm, _ = sc.to_local(ST['thx'], ST['thy'], ST['thz'], phin)
module = ST['tmodule'].astype(np.int64)
z_um = (w_mm - sc.si_layer_centers()[layer]) * 1e3 + THICK / 2    # depth in the slab [um]
faces = sc.module_face_phi(np.arange(sc.N_FACES))
AXIS_MODULE = int(np.flatnonzero(np.isclose(np.cos(faces - np.pi / 2), 1.0))[0])   # its normal points along +y
tol = sc.SLAB_TOL_MM * 1e3
assert np.array_equal(layer, ST['tlayer']), "the frame's layer is not the cellID layer"
assert np.all((z_um >= -tol) & (z_um <= THICK + tol)), "a step lies outside its slab"
print(f"check: every step lies inside the slab of its layer: depth z from {z_um.min():.3f} to {z_um.max():.3f} um")
mods, n_mod = np.unique(module, return_counts=True)
print("steps per module: " + ", ".join(f"module {m}: {n:,}" for m, n in zip(mods, n_mod)))
beam = [str(m) for m in ST['meta'] if 'beam' in str(m)]
print(f"the gun: {', '.join(beam)}; the module whose normal points along +y is module {AXIS_MODULE}")

Nearly every step is on the module that faces the beam. Below: the share of the deposited energy
and of the steps in each layer (the shower's depth profile, as the silicon sees it), and the
length and the deposit of the steps. The grey line marks where the layers move farther apart
(the printed pitches).

In [ ]:
n_lay = np.bincount(layer, minlength=N_LAYERS)
e_lay = np.bincount(layer, edep_eV, N_LAYERS) / 1e3                 # keV
photon = step_pdg == 22
at_limit = np.abs(path_um - STEP_LIMIT) < 1e-3
w_pitch = np.diff(sc.si_layer_centers())                            # mm from one layer to the next
L_WIDE = int(np.flatnonzero(w_pitch > w_pitch[0] + 1e-6)[0]) + 1    # the first layer after a wider gap
print(f"layer pitch: {w_pitch[0]:.2f} mm up to layer {L_WIDE - 1}, then {w_pitch[L_WIDE - 1]:.2f} mm "
      f"(the same to the last layer: {bool(np.allclose(w_pitch[L_WIDE - 1:], w_pitch[L_WIDE - 1]))})")

fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
ax = axs[0]
ax.step(np.arange(N_LAYERS), e_lay / e_lay.sum(), where='mid', color='C0', lw=2, label='deposited energy')
ax.step(np.arange(N_LAYERS), n_lay / n_lay.sum(), where='mid', color='C1', ls='--', lw=1.5, label='steps')
ax.axvline(L_WIDE - 0.5, color='0.6', lw=1, label='wider layer pitch from here')
ax.set_xlabel('layer')
ax.set_ylabel('share per layer')
ax.legend(fontsize=9)
ax = axs[1]
bins = np.logspace(-5, 3, 81)
ax.hist(path_um[~photon], bins=bins, histtype='stepfilled', color='C0', alpha=0.6, label='charged particles')
ax.hist(path_um[photon], bins=bins, histtype='step', color='C3', lw=1.5, label='photons')
ax.axvline(STEP_LIMIT, color='k', ls=':', label=f'step limit {STEP_LIMIT:g} um')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('step length [um]')
ax.set_ylabel('steps')
ax.legend(fontsize=9, loc='upper left')
ax = axs[2]
bins = np.logspace(-2, 5, 71)
ax.hist(edep_eV, bins=bins, histtype='stepfilled', color='C2', alpha=0.6, label='all steps')
ax.hist(edep_eV[at_limit], bins=bins, histtype='step', color='k', lw=1.2, label=f'steps of {STEP_LIMIT:g} um')
ax.axvline(EV_PER_PAIR, color='C3', ls=':', label=f'{EV_PER_PAIR} eV, one pair')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('deposit per step [eV]')
ax.set_ylabel('steps')
ax.legend(fontsize=9, loc='upper left')
fig.suptitle(f'shower {EVENT}: {n_step:,} steps in the silicon')
plt.show()

print(f"most energy in layer {int(np.argmax(e_lay))} ({e_lay.max():,.0f} keV, {int(n_lay[np.argmax(e_lay)]):,} steps); "
      f"layers without a step: {[int(l) for l in np.flatnonzero(n_lay == 0)]}")
print(f"steps at the {STEP_LIMIT:g} um limit: {100 * at_limit.mean():.1f}%; median length {np.median(path_um):.2f} um; "
      f"{100 * np.mean(path_um < 1):.1f}% are shorter than 1 um")
long_ = path_um > STEP_LIMIT + 1e-3
print(f"longer than {STEP_LIMIT:g} um: {int(long_.sum())} steps, all of photons: {bool(np.all(photon[long_]))} "
      f"(longest {path_um.max():.0f} um)")
print(f"deposit per step: median {np.median(edep_eV):,.0f} eV; for the {STEP_LIMIT:g} um steps median "
      f"{np.median(edep_eV[at_limit]):,.0f} eV ({np.median(edep_eV[at_limit]) / EV_PER_PAIR:,.0f} pairs), mean "
      f"{edep_eV[at_limit].mean():,.0f} eV ({edep_eV[at_limit].mean() / EV_PER_PAIR:,.0f} pairs); "
      f"{100 * np.mean(edep_eV < EV_PER_PAIR):.2f}% of all steps leave less than one pair's worth")

The deposit follows the shower: it rises over the first layers, peaks at layer 10 and then falls,
faster per layer from layer 20 on, where the layers are farther apart. The number of steps
follows the energy closely. About three quarters of the steps stop at the 5 um limit, and all
but a handful of the rest are shorter; only photon steps are longer.
A 5 um step of a charged particle leaves a few hundred pairs, with a wide spread: the mean lies
above the median (the printed values).

**Check against the jobs table.** `deposit_jobs.py` puts every step of the batch into exactly
one group, so over the groups of shower 2 its table must hold the same energy and the same number
of steps.

In [ ]:
g2 = JB['group_evt_id'] == EVENT
G0, NG = int(np.flatnonzero(g2)[0]), int(g2.sum())
assert np.array_equal(np.flatnonzero(g2), G0 + np.arange(NG)), "the groups of the shower are not consecutive rows"
assert np.array_equal(JB['group_id'], np.arange(len(JB['group_id']))), "group_id is not the row index"
e_groups = float(JB['group_edep_keV'][g2].sum())
rel = abs(e_groups - E_TOTAL_KEV) / E_TOTAL_KEV
assert rel <= 1e-9, f"the groups hold {e_groups:.6f} keV, the steps {E_TOTAL_KEV:.6f} keV"
assert int(JB['group_n_steps'][g2].sum()) == n_step, "the groups do not hold every step"
print(f"check: the {NG:,} groups of shower {EVENT} (group ids {G0:,} to {G0 + NG - 1:,}) hold {e_groups:,.3f} keV "
      f"(relative difference {rel:.1e}) and {int(JB['group_n_steps'][g2].sum()):,} steps, as the step file")

## 2. One particle through one slab

Call the steps of one particle in one sensor plane a *node*. Most particles make one node; some
cross the tungsten and make nodes in several planes (the printed count).

We pick a node that shows two things at once. Its particle crosses the slab nearly at right
angles: it enters through one face and leaves through the other, on a path at most 3% longer than
the thickness (a *normal crossing*). And on the way it knocks out exactly one delta ray, which
stays in the slab. The delta ray may knock out electrons of its own; we draw them with it.

To draw a step we use `deposit_jobs.segments`: each step becomes a straight segment centred on
its midpoint, along its mean momentum, as long as the step, and clipped in depth to the slab.
Section 4 cuts these segments into the pieces that PIXELAV receives.

In [ ]:
S, P, _, _ = dj._load(need('steps_b0_evt2.npz'), None)   # the arrays as deposit_jobs.py reads them
SEG = dj.segments(S, THICK)                                 # per step: the clipped segment, in um
assert np.allclose(SEG['u'], u_mm * 1e3) and np.allclose(SEG['v'], v_mm * 1e3), "segments use another frame"
END0 = {c: SEG[c] + SEG['d' + c] * SEG['s_lo'] for c in ('u', 'v', 'z')}   # the two ends of each segment
END1 = {c: SEG[c] + SEG['d' + c] * SEG['s_hi'] for c in ('u', 'v', 'z')}
z_lo, z_hi = np.minimum(END0['z'], END1['z']), np.maximum(END0['z'], END1['z'])

slab = module * N_LAYERS + layer
nodes, node_of_step = np.unique(tmc * N_SLABS + slab, return_inverse=True)
node_part, node_slab = nodes // N_SLABS, nodes % N_SLABS
node_mod, node_lay = node_slab // N_LAYERS, node_slab % N_LAYERS
N_NODES = len(nodes)
node_e = np.bincount(node_of_step, edep_eV, N_NODES)          # eV
node_path = np.bincount(node_of_step, path_um, N_NODES)       # um
node_zmin = np.full(N_NODES, np.inf)
node_zmax = np.full(N_NODES, -np.inf)
np.minimum.at(node_zmin, node_of_step, z_lo)
np.maximum.at(node_zmax, node_of_step, z_hi)
# was the particle made inside this slab? its creation point projected on the module's normal
phi_node = sc.module_face_phi(node_mod)
w_vtx = ST['vsx'][node_part] * np.cos(phi_node) + ST['vsy'][node_part] * np.sin(phi_node)
made_here = np.abs(w_vtx - sc.si_layer_centers()[node_lay]) <= THICK / 2e3 + sc.SLAB_TOL_MM
# the node of the particle's parent in the same slab, -1 when the parent has no steps there
want = np.where(parent[node_part] >= 0, parent[node_part] * N_SLABS + node_slab, -1)
at = np.minimum(np.searchsorted(nodes, want), N_NODES - 1)
mother = np.where((want >= 0) & (nodes[at] == want), at, -1)
crosses = (node_zmin < 1) & (node_zmax > THICK - 1)            # from one face to the other, within 1 um
normal_node = crosses & (node_path <= NORMAL * THICK)
per_part = np.bincount(node_part, minlength=n_part)
print(f"{N_NODES:,} nodes: {int((~made_here).sum()):,} particles entered their slab from outside, "
      f"{int(made_here.sum()):,} were made inside it (creation point within {THICK / 2:g} + "
      f"{sc.SLAB_TOL_MM * 1e3:g} um of the slab's mid-plane)")
print(f"particles with nodes in more than one plane: {int(np.sum(per_part > 1)):,} (at most {per_part.max()} planes)")
print(f"{int(crosses.sum()):,} nodes cross the slab from face to face, {int(normal_node.sum()):,} of them as normal crossings")

In [ ]:
kids = np.bincount(mother[made_here & (mother >= 0)], minlength=N_NODES)    # particles made from it in the slab
only_kid = np.full(N_NODES, -1)
sel = made_here & (mother >= 0)
only_kid[mother[sel]] = np.flatnonzero(sel)                                  # one of them (the only one when kids == 1)
ke_node = (ST['E'][node_part] - ST['mass'][node_part]) * 1e6                 # keV, kinetic energy where it was made
child_of = np.where(made_here & ST['parent_charged'][node_part], mother, -1)  # the node it was knocked out of


def family(n):
    # node n with the particles it knocked out in the slab, the ones those knocked out, and so on
    fam = np.array([n])
    while True:
        new = np.setdiff1d(np.flatnonzero(np.isin(child_of, fam)), fam)
        if len(new) == 0:
            return fam
        fam = np.r_[fam, new]


cand = np.flatnonzero(normal_node & ~made_here & (kids == 1) & (np.abs(ST['pdg'][node_part]) == 11))
cand = cand[np.abs(ST['pdg'][node_part[only_kid[cand]]]) == 11]
fams = {int(n): family(int(only_kid[n])) for n in cand}
stays = [n for n in fams if ke_node[only_kid[n]] - node_e[fams[n]].sum() / 1e3 < 0.01]   # its energy stays in the slab
N_ROOT = max(stays, key=lambda n: node_path[fams[n]].sum())                  # the delta ray with the longest path
N_DELTA, FAM = int(only_kid[N_ROOT]), fams[N_ROOT]
print(f"{len(cand)} normal crossings knock out exactly one delta ray in the slab; for {len(stays)} of them the delta "
      "ray leaves all its energy in the slab; the one whose delta ray has the longest path is shown")

ks = {'root': np.flatnonzero(node_of_step == N_ROOT), 'delta ray': np.flatnonzero(np.isin(node_of_step, FAM))}
v0 = np.average(SEG['v'][ks['root']], weights=edep_eV[ks['root']])
u0 = np.average(SEG['u'][ks['root']], weights=edep_eV[ks['root']])
k_all = np.concatenate(list(ks.values()))
e_all = edep_eV[k_all] / 1e3
cnorm = LogNorm(vmin=max(e_all.min(), 0.01), vmax=e_all.max())
half = max(20.0, np.ceil(max(np.abs(np.r_[END0['v'][k_all], END1['v'][k_all]] - v0).max(),
                             np.abs(np.r_[END0['u'][k_all], END1['u'][k_all]] - u0).max())) + 5)
fig, axs = plt.subplots(1, 2, figsize=(min(14.0, 11 * half / 55 + 3), 6.8), sharey=True, constrained_layout=True)
for ax, c, x0 in zip(axs, ('v', 'u'), (v0, u0)):
    for (name, k), mk, sz in zip(ks.items(), ('o', '^'), (36, 50)):
        for a, b, za, zb in zip(END0[c][k] - x0, END1[c][k] - x0, END0['z'][k], END1['z'][k]):
            ax.plot([a, b], [za, zb], color='0.55', lw=0.8, zorder=1)
        im = ax.scatter(SEG[c][k] - x0, SEG['z'][k], c=edep_eV[k] / 1e3, norm=cnorm, cmap='viridis', marker=mk,
                        s=sz, edgecolors='k', linewidths=0.4, zorder=2, label=name)
    for zf in (0, THICK):
        ax.axhline(zf, color='k', lw=1)
    ax.set_xlim(-half, half)
    ax.set_ylim(-5, THICK + 5)
    ax.set_aspect('equal')
    ax.set_xlabel(f'{c} - {c} of the track [um]')
    ax.set_title(f'section along {c}', fontsize=10)
axs[0].set_ylabel('depth z in the slab [um]')
axs[0].legend(loc='upper left', fontsize=8)
fig.colorbar(im, ax=axs, label='deposit of the step [keV]', shrink=0.8)
pn, ln = int(node_part[N_ROOT]), int(node_lay[N_ROOT])
fig.suptitle(f'shower {EVENT}, module {int(node_mod[N_ROOT])}, layer {ln}: particle row {pn} and its delta ray\n'
             'markers: step midpoints; grey lines: the steps as segments', fontsize=10)
plt.show()

In [ ]:
pd = int(node_part[N_DELTA])
ke_delta = float(ke_node[N_DELTA])                                               # keV
fam_e = node_e[FAM].sum() / 1e3                                                  # keV
z_made = (w_vtx[N_DELTA] - sc.si_layer_centers()[ln]) * 1e3 + THICK / 2
k_root, k_delta = ks['root'], ks['delta ray']
print(f"root: {pdg_name(ST['pdg'][pn])}, {SEG['p_MeV'][k_root[np.argmin(ST['ttime'][k_root])]]:.1f} MeV/c at its "
      f"first step in the slab; {len(k_root)} steps, path {node_path[N_ROOT]:.1f} um, depth {node_zmin[N_ROOT]:.2f} to "
      f"{node_zmax[N_ROOT]:.2f} um, deposit {node_e[N_ROOT] / 1e3:.2f} keV")
print(f"delta ray: {pdg_name(ST['pdg'][pd])} made at depth {z_made:.1f} um with {ke_delta:.2f} keV of kinetic energy; "
      f"with the {len(FAM) - 1} particles it knocked out in turn: {len(k_delta)} steps, path {node_path[FAM].sum():.1f} um, "
      f"deposit {fam_e:.2f} keV (kinetic energy - deposit: {ke_delta - fam_e:+.3f} keV)")
tot = node_e[N_ROOT] / 1e3 + fam_e
print(f"together {tot:.2f} keV = {tot * 1e3 / EV_PER_PAIR:,.0f} pairs; the delta ray holds {100 * fam_e / tot:.0f}% of it, "
      f"{fam_e / (node_e[N_ROOT] / 1e3):.1f} x the root's own deposit")

The root crosses the slab almost straight, losing a little energy in every step. The delta ray
starts on the track, wanders off sideways and stops inside the slab: its deposit equals the
kinetic energy it was made with, and it is about four times what the root left along the whole
crossing (the printed values).

**The textbook.** A minimum ionising particle (MIP) is a charged particle fast enough that its
energy loss per unit length sits near the minimum. As a rough guide: a MIP loses on average
about 1.66 MeV cm²/g x 2.33 g/cm³ = 3.87 MeV/cm in silicon, so 100 um takes about 38.7 keV, which
is about 10,510 pairs at 3.68 eV each. In a layer this thin the energy loss has a long tail to
high values, and its most probable value is only about 70% of the mean, roughly 7,357 e. Both
values are approximate. A crossing with one large delta ray, like the one above, sits in that
tail. Notebook 10 (section 4) uses the same numbers.

In [ ]:
DEDX_MIN = 1.66      # MeV cm2/g, minimum ionisation in silicon (textbook value)
RHO_SI = 2.33        # g/cm3
mip_mean_keV = DEDX_MIN * RHO_SI * (THICK * 1e-4) * 1e3
mip_mean_e = mip_mean_keV * 1e3 / EV_PER_PAIR
mip_mpv_e = 0.7 * mip_mean_e
print(f"textbook: {DEDX_MIN} MeV cm2/g x {RHO_SI} g/cm3 = {DEDX_MIN * RHO_SI:.2f} MeV/cm; {THICK:g} um take "
      f"{mip_mean_keV:.1f} keV = {mip_mean_e:,.0f} pairs at {EV_PER_PAIR} eV")
print(f"textbook (approximate): mean {mip_mean_e:,.0f} e, most probable about {mip_mpv_e:,.0f} e for {THICK:g} um "
      f"(70% of the mean)")
print(f"the particle above: {tot * 1e3 / EV_PER_PAIR / mip_mean_e:.2f} x the textbook mean")

## 3. Groups

PIXELAV moves charge, and charge has no memory of which particle made it. What the study counts
is a particle crossing a plane, with the delta rays it made on the way. `deposit_jobs.py` (its
docstring, GROUPS) joins nodes into *groups* by one rule:

- a node joins its parent's node in the same plane when the particle was made inside that slab
  (its creation point lies within half the thickness of the slab's mid-plane, plus a tolerance of
  2 um) and the parent is charged and has steps in that plane;
- following the joins upward ends at a node that joins nothing: the group's *root*;
- `created_in_slab` marks a root that was itself made inside the slab (for example an electron
  from a photon that converted or scattered in the silicon).

So a delta ray joins the track that made it, and a photon converting in the silicon gives two
roots, because a photon is not charged. Below we apply the rule with a plain loop over the nodes
of section 2, then compare with `deposit_jobs.groups` and with the groups of the batch's table.

In [ ]:
up = np.arange(N_NODES)                                              # the node each node joins (itself: none)
joins = (mother >= 0) & made_here & ST['parent_charged'][node_part]
up[joins] = mother[joins]
root_node = np.empty(N_NODES, np.int64)
for i in range(N_NODES):                                             # follow the joins to the top
    j = i
    while up[j] != j:
        j = up[j]
    root_node[i] = j
roots, group_of_node = np.unique(root_node, return_inverse=True)     # groups in the order of their root node
group_of_step = group_of_node[node_of_step]
assert len(roots) == NG, f"{len(roots)} groups, the table has {NG}"

G = dj.groups(S, P, THICK / 1e3)
assert np.array_equal(G['step_group'], group_of_step), "another grouping of the steps than deposit_jobs.groups"
assert np.array_equal(G['root'], node_part[roots]), "other roots than deposit_jobs.groups"
print(f"check: the loop gives the same {len(roots):,} groups and roots as deposit_jobs.groups, step by step")

mine = {'group_root': node_part[roots] + FIRST_PART, 'group_module': node_mod[roots], 'group_layer': node_lay[roots],
        'group_n_particles': np.bincount(group_of_node, minlength=NG),
        'group_n_steps': np.bincount(group_of_step, minlength=NG),
        'group_edep_keV': np.bincount(group_of_step, edep_eV, NG) / 1e3,
        'group_created_in_slab': made_here[roots]}
for k, v in mine.items():
    assert np.array_equal(v, JB[k][G0:G0 + NG]), f"{k} differs from the batch's table"
print(f"check: {', '.join(mine)} equal the batch's table for the {NG:,} groups of shower {EVENT}")

In [ ]:
GR = {k: JB[k][G0:G0 + NG] for k in JB if k.startswith('group_')}   # the groups of shower 2, from the table
GR.update({k: CH[k][G0:G0 + NG] for k in ('group_q', 'group_n_pix')})
is_root_step = tmc == node_part[roots][group_of_step]
nonroot_share = edep_eV[~is_root_step].sum() / edep_eV.sum()
rt, rn = np.unique(GR['group_root_pdg'], return_counts=True)
npg = GR['group_n_particles']
print("roots by type: " + ", ".join(f"{pdg_name(t)} {n:,}" for t, n in sorted(zip(rt, rn), key=lambda x: -x[1])))
print(f"particles per group: {100 * np.mean(npg == 1):.1f}% hold one, median {np.median(npg):.0f}, "
      f"{100 * np.mean(npg > 10):.1f}% more than 10, largest {npg.max()}")
print(f"roots made inside their slab (created_in_slab): {int(GR['group_created_in_slab'].sum())} "
      f"({100 * GR['group_created_in_slab'].mean():.1f}%)")
print(f"share of the shower's deposit made by particles other than the roots (delta rays and their own delta rays): "
      f"{100 * nonroot_share:.1f}%")

g_dep = np.bincount(group_of_node, node_e, NG)                     # eV per group
dshare = 1 - np.divide(node_e[roots], g_dep, out=np.ones(NG), where=g_dep > 0)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
ax = axs[0]
ax.hist(npg, bins=np.arange(0.5, npg.max() + 1.5), histtype='stepfilled', color='C0', alpha=0.7)
ax.set_xlim(0, npg.max() + 2)
ax.set_yscale('log')
ax.set_xlabel('particles in the group')
ax.set_ylabel('groups')
ax = axs[1]
ax.hist(dshare[npg > 1], bins=np.linspace(0, 1, 41), histtype='stepfilled', color='C1', alpha=0.7)
ax.set_xlabel("share of the group's deposit made by its delta rays")
ax.set_ylabel('groups with at least one delta ray')
fig.suptitle(f'the {NG:,} groups of shower {EVENT}')
plt.show()

Most groups are one particle with a few delta rays. The delta rays make a sizeable share of the
deposit, so leaving them out, or counting each as a particle of its own, would change both the
charge and the count. A group is the object whose charge the next sections follow.

## 4. Jobs and the window frame

PIXELAV computes the charge on a window of 21 x 13 pixels. A shower spreads over centimetres,
so each sensor plane is cut into windows. `deposit_jobs.py` (TILES and JOBS in its docstring)
does it like this:

- every plane has one pixel grid: pixel `iv = floor(v / 50 um)`, `iu = floor(u / 25 um)`;
- the grid is covered by *tiles* of 17 x 9 pixels, tile `tv = floor(iv / 17)`, `tu = floor(iu / 9)`;
- each step's segment is cut into equal *pieces* of at most 5 um, each with its share of the
  energy by length. A piece belongs to the tile that holds its midpoint;
- a *job* is one group in one tile; its window is the tile plus a guard band of 2 pixels on
  every side. The band holds the half piece that may reach past the tile and the charge that
  spreads sideways while it drifts;
- inside a window, PIXELAV's frame has `x` along `v` and `y` along `u`, with the centre of window
  pixel (10, 6) at (0, 0):

  `x = v - (tv x 17 + 8 + 0.5) x 50 um`,   `y = u - (tu x 9 + 4 + 0.5) x 25 um`,

  and `z` from 0 on the inner face to 100 um on the outer one;
- a job that would make more than 100,000 pairs is split into several jobs with the same window.

First we run `deposit_jobs.py` on shower 2, as a script would, into a temporary directory, and
compare its tables with the batch's.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    prefix = os.path.join(tmp, f'evt{EVENT}')
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        rc = dj.main([need('steps_b0_evt2.npz'), prefix, '--pitch-x-um', f'{PITCH_V:g}', '--pitch-y-um', f'{PITCH_U:g}',
                      '--thick-um', f'{THICK:g}'])
    with np.load(prefix + '.jobs.npz', allow_pickle=False) as d:
        RB = {k: d[k] for k in d.files}
    TEXT = np.loadtxt(prefix + '.jobs.txt', comments='J', ndmin=2)   # the pieces as PIXELAV reads them
assert rc == 0, "deposit_jobs.py failed"
print(out.getvalue().replace(tmp, '<temporary directory>'))

In [ ]:
j2 = JB['job_evt_id'] == EVENT
J0, NJ = int(np.flatnonzero(j2)[0]), int(j2.sum())
assert np.array_equal(np.flatnonzero(j2), J0 + np.arange(NJ)), "the jobs of the shower are not consecutive rows"
assert np.array_equal(RB['group_id'] + G0, JB['group_id'][G0:G0 + NG]), "other group ids"
assert np.array_equal(RB['job_id'] + J0, JB['job_id'][J0:J0 + NJ]), "other job ids"
offset = {'group_root': FIRST_PART, 'job_group_id': G0}
cols = [k for k in RB if k.startswith(('group_', 'job_')) and k not in ('group_id', 'job_id', 'job_part')]
for k in cols:
    rows = slice(G0, G0 + NG) if k.startswith('group_') else slice(J0, J0 + NJ)
    assert np.array_equal(RB[k] + offset.get(k, 0), JB[k][rows]), f"{k} differs from the batch's table"
n_g = sum(k.startswith('group_') for k in cols)
print(f"check: all {n_g} group columns and {len(cols) - n_g} job columns equal the batch's for shower {EVENT}, "
      f"ids shifted by {G0:,} (groups) and {J0:,} (jobs), group_root by {FIRST_PART:,}")
print(f"(job_part is left out: it names the part file, {int(JB['parts'])} parts for the batch, 1 here; "
      f"the batch's jobs of this shower sit in parts {int(JB['job_part'][J0])} to {int(JB['job_part'][J0 + NJ - 1])})")

The script reproduces the batch exactly. To see the pieces themselves, we call the function that
builds them, `deposit_jobs.build_block`, and then cut the segments into pieces and move them into
the window frame ourselves, with the formula above.

In [ ]:
SENSOR = dj.sensor_facts(None, PITCH_V, PITCH_U, THICK)
PCS, JOBS, _ = dj.build_block(S, P, SENSOR, THICK, THICK / 1e3, PIECE)    # pieces in job order, window frame
assert np.array_equal(JOBS['n_seg'], RB['job_n_seg']) and np.allclose(JOBS['edep_keV'], RB['job_edep_keV'], rtol=1e-12)

# our own pieces: cut every segment into n equal pieces of at most PIECE um
length = SEG['s_hi'] - SEG['s_lo']
n_pc = np.maximum(1, np.ceil(length / PIECE - 1e-9)).astype(np.int64)
st = np.repeat(np.arange(n_step), n_pc)                            # the step of every piece
k = np.arange(n_pc.sum()) - np.repeat(np.cumsum(n_pc) - n_pc, n_pc)
dl = (length / n_pc)[st]
s0 = SEG['s_lo'][st] + k * dl
e_pc = (edep_eV / n_pc)[st]                                        # energy shared by length
A = {c: SEG[c][st] + SEG['d' + c][st] * s0 for c in ('u', 'v', 'z')}
B = {c: SEG[c][st] + SEG['d' + c][st] * (s0 + dl) for c in ('u', 'v', 'z')}
tv = np.floor((A['v'] + B['v']) / 2 / PITCH_V).astype(np.int64) // TILE_X
tu = np.floor((A['u'] + B['u']) / 2 / PITCH_U).astype(np.int64) // TILE_Y
cx = (tv * TILE_X + (CENTRE_X - GUARD) + 0.5) * PITCH_V            # the window formula
cy = (tu * TILE_Y + (CENTRE_Y - GUARD) + 0.5) * PITCH_U
mine = {'x0': A['v'] - cx, 'y0': A['u'] - cy, 'z0': np.clip(A['z'], 0, THICK),
        'x1': B['v'] - cx, 'y1': B['u'] - cy, 'z1': np.clip(B['z'], 0, THICK), 'e': e_pc}
order = np.lexsort((np.arange(len(st)), tu, tv, group_of_step[st]))  # job order: group, tile, then piece
for c, a in mine.items():
    assert np.allclose(a[order], PCS[c], rtol=0, atol=1e-6), f"our pieces differ from build_block's in {c}"
print(f"window formula: x = v - (tv x {TILE_X} + {CENTRE_X - GUARD} + 0.5) x {PITCH_V:g} um, "
      f"y = u - (tu x {TILE_Y} + {CENTRE_Y - GUARD} + 0.5) x {PITCH_U:g} um")
print(f"check: our {len(st):,} pieces from {n_step:,} steps equal build_block's in all seven numbers "
      f"(x0, y0, z0, x1, y1, z1 within 1e-6 um, energy within 1e-6 eV)")
assert np.allclose(TEXT, np.column_stack([PCS[c] for c in ('x0', 'y0', 'z0', 'x1', 'y1', 'z1', 'e')]),
                   rtol=0, atol=0.0051), "the jobs text is not the pieces"
print(f"check: the {len(TEXT):,} lines of the jobs text are these pieces, to the 3 decimals (um) and 2 (eV) written")

In [ ]:
diff, n_outside = dj.check_block(PCS, JOBS, edep_eV.sum(), THICK, SENSOR, PIECE)
assert abs(diff) <= 1e-9 * edep_eV.sum(), f"the pieces hold {diff:+.3g} eV more than the steps"
assert n_outside == 0, f"{n_outside} pieces lie outside their tile"
bx, by = TILE_X / 2 * PITCH_V + PIECE / 2, TILE_Y / 2 * PITCH_U + PIECE / 2
inside = all(np.all(np.abs(PCS[c]) <= b + 1e-6) for c, b in (('x0', bx), ('x1', bx), ('y0', by), ('y1', by)))
assert inside, "a piece reaches more than half a piece past its tile"
print(f"check: energy closure, pieces {PCS['e'].sum() / 1e3:,.6f} keV against steps {E_TOTAL_KEV:,.6f} keV "
      f"(difference {diff:+.2e} eV)")
print(f"check: every piece end lies within its tile plus half a piece: |x| <= {bx:g} um, |y| <= {by:g} um "
      f"(the tile is {TILE_X * PITCH_V:g} x {TILE_Y * PITCH_U:g} um)")
far_x = np.maximum(np.abs(PCS['x0']), np.abs(PCS['x1'])).max() - TILE_X / 2 * PITCH_V
far_y = np.maximum(np.abs(PCS['y0']), np.abs(PCS['y1'])).max() - TILE_Y / 2 * PITCH_U
print(f"farthest reach past the tile edge: {far_x:.2f} um along x, {far_y:.2f} um along y")

jobs_per_group = np.bincount(RB['job_group_id'], minlength=NG)
pairs_job = RB['job_edep_keV'] * 1e3 / EV_PER_PAIR
split_b = JB['job_n_split'] > 1
print(f"\njobs: {NJ:,} for {NG:,} groups; groups in one tile {100 * np.mean(GR['group_n_tiles'] == 1):.1f}%, in two "
      f"{100 * np.mean(GR['group_n_tiles'] == 2):.1f}%, most {GR['group_n_tiles'].max()} tiles")
print(f"pieces per job: median {np.median(RB['job_n_seg']):.0f}, most {RB['job_n_seg'].max()}; pairs per job: median "
      f"{np.median(pairs_job):,.0f}, most {pairs_job.max():,.0f}, against the cap of {int(JB['max_pairs']):,}")
print(f"shower {EVENT} needs no split (every group has one job per tile: "
      f"{bool(np.array_equal(jobs_per_group, GR['group_n_tiles']))}); in the whole batch "
      f"{int(np.round((1 / JB['job_n_split'][split_b]).sum()))} (group, tile) pairs were split into "
      f"{int(split_b.sum())} jobs, at most {JB['job_n_split'].max()} each")

**A group that spans two tiles.** We draw one particle whose track crosses a tile edge: its
pieces go to two jobs, each with its own window. The windows overlap; PIXELAV runs each job on
its own, and `charge_maps.py` adds the two charge maps on the plane's pixel grid.

In [ ]:
gsel = np.flatnonzero((GR['group_n_tiles'] == 2) & (GR['group_n_particles'] == 1) &
                      (GR['group_path_um'] > 120) & (GR['group_path_um'] < 300))
first_job = np.searchsorted(RB['job_group_id'], gsel)
e_two = np.column_stack([RB['job_edep_keV'][first_job], RB['job_edep_keV'][first_job + 1]])
G_TWO = int(gsel[np.argmin(np.abs(e_two[:, 0] - e_two[:, 1]) / e_two.sum(axis=1))])   # the most even split
jj = np.flatnonzero(RB['job_group_id'] == G_TWO)
pc_start = np.r_[0, np.cumsum(RB['job_n_seg'])]
assert np.array_equal(pc_start[:-1], JOBS['start']), "the jobs' pieces are not consecutive"
print(f"rule: among the {len(gsel)} groups of one particle in two tiles with a path of 120 to 300 um, the one whose "
      f"energy is shared most evenly between its two jobs: group {G0 + G_TWO:,}")
fig, (ax, az) = plt.subplots(1, 2, figsize=(14, 6.2), constrained_layout=True, gridspec_kw={'width_ratios': [2.3, 1]})
lo_v, hi_v, lo_u, hi_u = np.inf, -np.inf, np.inf, -np.inf
for j, col in zip(jj, ('C0', 'C3')):
    tvj, tuj = int(RB['job_tile_v'][j]), int(RB['job_tile_u'][j])
    iv0, iu0 = int(RB['job_win_iv0'][j]), int(RB['job_win_iu0'][j])
    for a_, lw in ((ax, 1.5), (az, 1.2)):
        a_.add_patch(Rectangle((iv0 * PITCH_V, iu0 * PITCH_U), WIN_X * PITCH_V, WIN_Y * PITCH_U, fill=False, ec=col,
                               ls='--', lw=lw, label=f'job {J0 + j:,}: window of tile ({tvj}, {tuj})'))
        a_.add_patch(Rectangle((tvj * TILE_X * PITCH_V, tuj * TILE_Y * PITCH_U), TILE_X * PITCH_V, TILE_Y * PITCH_U,
                               fill=True, fc=col, alpha=0.08, ec=col, lw=lw, label=f'tile ({tvj}, {tuj})'))
    p = slice(pc_start[j], pc_start[j + 1])
    cxj = (tvj * TILE_X - GUARD + CENTRE_X + 0.5) * PITCH_V           # the window formula, back to the plane
    cyj = (tuj * TILE_Y - GUARD + CENTRE_Y + 0.5) * PITCH_U
    xa, xb_, ya, yb = PCS['x0'][p] + cxj, PCS['x1'][p] + cxj, PCS['y0'][p] + cyj, PCS['y1'][p] + cyj
    for a, b, c_, d_ in zip(xa, xb_, ya, yb):
        ax.plot([a, b], [c_, d_], color=col, lw=2.5, solid_capstyle='butt')
        az.plot([a, b], [c_, d_], color=col, lw=2, solid_capstyle='butt')
    az.plot(np.r_[xa, xb_], np.r_[ya, yb], '.', color='k', ms=3)
    ax.plot([], [], color=col, lw=2.5, label=f'its {pc_start[j + 1] - pc_start[j]} pieces, {RB["job_edep_keV"][j]:.1f} keV')
    lo_v, hi_v = min(lo_v, xa.min(), xb_.min()), max(hi_v, xa.max(), xb_.max())
    lo_u, hi_u = min(lo_u, ya.min(), yb.min()), max(hi_u, ya.max(), yb.max())
iv_lo = min(int(RB['job_win_iv0'][j]) for j in jj)
iu_lo = min(int(RB['job_win_iu0'][j]) for j in jj)
iv_hi = max(int(RB['job_win_iv0'][j]) for j in jj) + WIN_X
iu_hi = max(int(RB['job_win_iu0'][j]) for j in jj) + WIN_Y
for a_ in (ax, az):
    for iv in range(iv_lo, iv_hi + 1):
        a_.axvline(iv * PITCH_V, color='0.85', lw=0.5, zorder=0)
    for iu in range(iu_lo, iu_hi + 1):
        a_.axhline(iu * PITCH_U, color='0.85', lw=0.5, zorder=0)
    a_.set_aspect('equal')
    a_.set_xlabel('v [um]')
    a_.set_ylabel('u [um]')
ax.set_xlim(iv_lo * PITCH_V - 20, iv_hi * PITCH_V + 20)
ax.set_ylim(iu_lo * PITCH_U - 20, iu_hi * PITCH_U + 20)
ax.add_patch(Rectangle((lo_v - 15, lo_u - 15), hi_v - lo_v + 30, hi_u - lo_u + 30, fill=False, ec='k', lw=0.8))
mid_v, mid_u, side = (lo_v + hi_v) / 2, (lo_u + hi_u) / 2, max(hi_v - lo_v, hi_u - lo_u) / 2 + 15
az.set_xlim(mid_v - side, mid_v + side)
az.set_ylim(mid_u - side, mid_u + side)
az.set_title('the black box, enlarged; dots: piece ends', fontsize=10)
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2)
ax.set_title(f'group {G0 + G_TWO:,}: one {pdg_name(GR["group_root_pdg"][G_TWO])} of '
             f'{GR["group_root_p_MeV"][G_TWO]:.1f} MeV/c, path {GR["group_path_um"][G_TWO]:.0f} um; grey lines: pixel edges',
             fontsize=10)
plt.show()

Each piece goes to the tile that holds its midpoint, so the track is cut at the tile edge. Each
window reaches 2 pixels past its tile on every side, so the two windows overlap. Section 5 checks
that no charge leaves a window.

## 5. PIXELAV's part

For every job, PIXELAV's injection driver (`ppixelav2_inject_deposits`, see the docstrings of
`charge_maps.py` and `deposit_jobs.py`, and section 5.3 of `docs/pixelav_handoff.md`):

1. makes the electron-hole pairs of each piece: with `n = energy / 3.68 eV`, it makes the whole
   part of `n`, plus one more pair with a probability equal to the remaining fraction. The pairs
   lie at random along the piece. Pairs made outside the 21 x 13 window are counted (`n_out`)
   and not moved;
2. moves every carrier through the sensor's electric field map: drift along the field, with
   random diffusion on the way. The field map is that of a 100 um thick pixel sensor with
   50 x 25 um pixels. There is no magnetic field;
3. records the charge on each pixel of the window at the end of the time it simulates (its last
   time slice), in electrons, with one decimal.

`charge_maps.py` then adds the maps of a group's jobs on the plane's pixel grid. Its output
keeps, per group and pixel, the charge `pix_q`; per job the pairs transported `job_neh`, the
pairs outside the window `job_n_out` and the window's charge `job_q`; per group its charge
`group_q` and its number of pixels `group_n_pix`.

First, what the charges file records about the run.

In [ ]:
print(f"sensor title (line 1 of the field map's ppixel2.init): {CH['pixelav_title']}")
print(f"sensor of the jobs: {CH['sensor_source']}")
print(f"time slice {float(CH['pixelav_stimstp_ps']):g} ps; {cm.NCRRC} slices, of which the driver wrote the last; "
      f"{len(CH['out_files'])} output files collected")
if 'pixelav_run' in CH:
    print("run record:\n" + str(CH['pixelav_run']))
else:
    print("run record (pixelav_run): not in this file, so the driver seed, the sha256 of the sensor files, the commits, "
          "the host and the wall time of this run are not recorded here")

The title line is the name of the field map, and it mentions a 3.8 T field. The sensor line
records the magnetic field the run used in the transport, B = (0, 0, 0) T. Section 7 checks this
from the charge itself: in a magnetic field the drifting charge is pushed sideways, without one it
lands straight below the track.

The charges file holds no record of the driver run (its seed, the checksums of the sensor files,
the code versions), so this notebook cannot print them.

**Checks over the whole batch.** We redo the checks of `charge_maps.py`, on all its jobs and
groups:

1. the pairs made, `job_neh + job_n_out`, equal the job's energy / 3.68 eV give or take one pair
   per piece (each piece makes the whole part of its pair count, or one more), plus the rounding
   of the energies in the jobs text (0.005 eV a piece);
2. no pair was made outside its window;
3. the pixel charges of each group add up to `group_q` (which was summed in double precision
   before the pixels were stored in single precision), and `group_n_pix` counts its pixels.

In [ ]:
made = CH['job_neh'] + CH['job_n_out']                               # pairs made, per job
expect = JB['job_edep_keV'] * 1e3 / EV_PER_PAIR                      # pairs expected from the energy
allowed = JB['job_n_seg'] * (1 + cm.SEG_ROUND_EV / EV_PER_PAIR) + 1e-6
bad = ~(np.abs(made - expect) < allowed)
assert not bad.any(), f"{int(bad.sum())} jobs made another number of pairs than energy / {EV_PER_PAIR} eV"
print(f"check 1: all {len(made):,} jobs made energy / {EV_PER_PAIR} eV pairs within one pair per piece; largest "
      f"|made - expected| {np.abs(made - expect).max():.1f} pairs, at most {np.max(np.abs(made - expect) / JB['job_n_seg']):.2f} "
      f"per piece; in total {made.sum():,} made against {expect.sum():,.0f} expected")
assert int(CH['job_n_out'].sum()) == 0 and int(CH['total_n_out']) == 0, "pairs were made outside their window"
print(f"check 2: pairs made outside their window: {int(CH['job_n_out'].sum())}")

pix_g = CH['pix_group_id'].astype(np.int64)
pix_iv, pix_iu = CH['pix_iv'].astype(np.int64), CH['pix_iu'].astype(np.int64)
pix_q = CH['pix_q'].astype(np.float64)
n_groups = len(CH['group_id'])
q_sum = np.bincount(pix_g, pix_q, n_groups)
q_tol = np.bincount(pix_g, np.abs(pix_q), n_groups) * 2.0 ** -24 + 1e-6     # the float32 rounding of the pixels
assert np.all(np.abs(q_sum - CH['group_q']) <= q_tol), "pixel charges do not add up to group_q"
assert np.array_equal(np.bincount(pix_g, minlength=n_groups), CH['group_n_pix']), "group_n_pix is not the pixel count"
assert np.all(np.diff(pix_g) >= 0), "the pixel rows are not sorted by group"
print(f"check 3: over {n_groups:,} groups and {len(pix_q):,} (group, pixel) rows, the pixels add up to group_q "
      f"(largest difference {np.abs(q_sum - CH['group_q']).max():.4f} e) and group_n_pix counts them")

**A stricter check on shower 2.** For the jobs of shower 2 we have the energy of every piece as
PIXELAV read it (the jobs text of section 4). So the pairs of a job must lie between the sum of
the whole parts of the pieces' pair counts and that sum plus the number of pieces with a fraction.
Beyond the bounds, the extra pairs are random: their spread is known, so the difference between
the pairs made and `energy / 3.68 eV`, divided by that spread, should scatter like a standard
normal variable (mean 0, standard deviation 1).

In [ ]:
n_e = TEXT[:, 6] / EV_PER_PAIR                                       # pairs expected from each piece
frac = n_e - np.floor(n_e)
job_of_piece = np.repeat(np.arange(NJ), RB['job_n_seg'])
lo = np.bincount(job_of_piece, np.floor(n_e), NJ)
hi = lo + np.bincount(job_of_piece, frac > 0, NJ)
made2 = made[J0:J0 + NJ]
assert np.all((made2 >= lo) & (made2 <= hi)), "pairs outside the bounds of the whole parts"
spread = np.sqrt(np.bincount(job_of_piece, frac * (1 - frac), NJ))
zs = (made2 - np.bincount(job_of_piece, n_e, NJ)) / np.maximum(spread, 1e-9)
assert abs(zs.mean()) < 5 / np.sqrt(NJ) and abs(zs.std() - 1) < 0.05, "the extra pairs do not scatter as expected"
print(f"check: every one of the {NJ:,} jobs of shower {EVENT} lies within its bounds; the extra pairs: "
      f"{np.mean(made2 - lo):.1f} per job on average")
print(f"check: (made - energy / {EV_PER_PAIR} eV) / spread: mean {zs.mean():+.3f}, standard deviation {zs.std():.3f} "
      f"(expected 0 and 1; the mean may move by {1 / np.sqrt(NJ):.3f})")

**What is lost, and where.** The charge PIXELAV collects on a window can differ from the pairs it
transported in a few ways: charge can spread out of the window, and the driver writes each pixel
with one decimal. We measure:

- the window's charge against the pairs transported, `job_q / job_neh`, per job;
- for groups in a single tile (one job, one window), the share of their charge that lands in the
  guard band, outside the tile;
- the pixels that hold less than 1 e, and the pixels with negative charge.

In [ ]:
neh, jq = CH['job_neh'], CH['job_q']
deficit = neh - jq
ratio_all = jq.sum() / neh.sum()
print(f"collected / transported over the batch: {ratio_all:.9f} ({jq.sum():,.1f} e of {neh.sum():,} pairs; "
      f"{deficit.sum():,.1f} e missing)")
print(f"per job, transported - collected: from {deficit.min():+.1f} to {deficit.max():+.1f} e; jobs that differ by more "
      f"than 0.5 e: {int(np.sum(np.abs(deficit) > 0.5))} of {len(neh):,}")

single = np.flatnonzero((CH['group_n_tiles'] == 1) & (np.bincount(JB['job_group_id'], minlength=n_groups) == 1))
job_of_group = np.full(n_groups, -1)
job_of_group[JB['job_group_id']] = np.arange(len(JB['job_id']))
jg = job_of_group[pix_g]
tile_v, tile_u = JB['job_tile_v'][jg], JB['job_tile_u'][jg]
in_tile = ((pix_iv // TILE_X) == tile_v) & ((pix_iu // TILE_Y) == tile_u)
guard_q = np.bincount(pix_g, pix_q * ~in_tile, n_groups)
share_guard = guard_q[single] / np.maximum(CH['group_q'][single], 1e-9)
print(f"groups in one tile with one job: {len(single):,}; share of their charge in the guard band: none for "
      f"{100 * np.mean(share_guard == 0):.1f}% of them, above 1% for {100 * np.mean(share_guard > 0.01):.2f}%, mean "
      f"{share_guard.mean():.2e}, largest {share_guard.max():.3f}")
small = pix_q < 1
print(f"pixels below 1 e: {int(small.sum())} of {len(pix_q):,} rows, together {pix_q[small].sum():.1f} e; "
      f"negative: {int(np.sum(pix_q < 0))}, smallest {pix_q.min():.2f} e")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
ax = axs[0]
xs = np.linspace(-4, 4, 161)
ax.hist(zs, bins=np.linspace(-4, 4, 41), density=True, histtype='stepfilled', color='C0', alpha=0.6,
        label=f'{NJ:,} jobs of shower {EVENT}')
ax.plot(xs, norm.pdf(xs), 'k--', label='standard normal')
ax.set_xlabel(f'(pairs made - energy / {EV_PER_PAIR} eV) / expected spread')
ax.set_ylabel('density')
ax.legend(fontsize=9)
ax = axs[1]
ax.hist(np.maximum(share_guard, 1e-6), bins=np.logspace(-6, 0, 61), histtype='stepfilled', color='C1', alpha=0.7)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('share of the charge in the guard band (values below 1e-6 drawn at 1e-6)')
ax.set_ylabel('groups in one tile')
plt.show()

Nothing measurable is lost. The pairs follow the rule of the driver. Every job collects the
pairs it transported to within one electron, so no measurable charge leaves a window. Some
groups do put charge into the guard band (the printed shares), and it is collected there. Pixels
below 1 e are a few dozen of the rows, and the negative ones are at most a tenth of an electron
below zero. From here on the charge of a group is its deposited energy / 3.68 eV, spread over the
pixels.

## 6. The charge of one crossing

Now we look at the charge of the simplest object: a group whose root is a normal crossing (it
enters through one face, leaves through the other, on a path at most 3% longer than the thickness)
and lies in one tile. We use the root's own path and depth range, from the steps of shower 2.

Why not the group's columns? `group_path_um` adds the paths of the delta rays to that of the root,
so a cut on it also removes the groups with long delta rays, which tend to have a large charge.
The cell compares the two choices (the printed medians).

The most probable charge is the peak of a parabola fitted to the logarithm of the counts in the
bins around the fullest bin of the charge histogram (the method of notebook 10); the printout
gives how much it moves with the bin width.

In [ ]:
rn_ = roots                                                # the root node of every group of shower 2
r_path, r_zmin, r_zmax = node_path[rn_], node_zmin[rn_], node_zmax[rn_]
one_tile = GR['group_n_tiles'] == 1
NORMAL_G = one_tile & (r_zmin < 1) & (r_zmax > THICK - 1) & (r_path <= NORMAL * THICK)
by_group_path = one_tile & (GR['group_w_min_um'] < 1) & (GR['group_w_max_um'] > THICK - 1) & \
                (GR['group_path_um'] <= NORMAL * THICK)
qg = GR['group_q']
print(f"normal crossings by the root's path: {int(NORMAL_G.sum()):,} groups, mean charge {qg[NORMAL_G].mean():,.0f} e, "
      f"median {np.median(qg[NORMAL_G]):,.0f} e")
print(f"by the group's path (root and delta rays): {int(by_group_path.sum()):,} groups, mean charge "
      f"{qg[by_group_path].mean():,.0f} e; the {int((NORMAL_G & ~by_group_path).sum())} normal crossings it drops have a "
      f"median charge of {np.median(qg[NORMAL_G & ~by_group_path]):,.0f} e")


def most_probable(x, width):
    # peak of a parabola fitted to the log of the counts in the 9 bins around the fullest one
    h, e = np.histogram(x, bins=np.arange(0, 30001, width))
    c = 0.5 * (e[1:] + e[:-1])
    j = int(np.argmax(h))
    s = slice(max(j - 4, 0), j + 5)
    p = np.polyfit(c[s], np.log(np.maximum(h[s], 1)), 2)
    return -p[1] / (2 * p[0])


qn = qg[NORMAL_G]
mpv_all = [most_probable(qn, w) for w in (100, 200, 250, 400)]
MPV = mpv_all[1]
q_root = node_e[rn_][NORMAL_G] / EV_PER_PAIR                       # the root's own deposit, as pairs
print(f"\ncharge of a normal crossing: most probable {MPV:,.0f} e (bins of 100 to 400 e give {min(mpv_all):,.0f} to "
      f"{max(mpv_all):,.0f} e), median {np.median(qn):,.0f} e, mean {qn.mean():,.0f} e")
print(f"textbook: most probable about {mip_mpv_e:,.0f} e, mean {mip_mean_e:,.0f} e; measured / textbook: most probable "
      f"{MPV / mip_mpv_e:.2f}, mean {qn.mean() / mip_mean_e:.2f}")
print(f"the root alone: median {np.median(q_root):,.0f} e, 95% of the crossings below {np.percentile(q_root, 95):,.0f} e; "
      f"the whole group: 95% below {np.percentile(qn, 95):,.0f} e")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4), constrained_layout=True)
ax = axs[0]
bins = np.arange(0, 40001, 500)
ax.hist(qn, bins=bins, histtype='stepfilled', color='C0', alpha=0.6, label='group (root and its delta rays)')
ax.hist(q_root, bins=bins, histtype='step', color='C2', lw=1.8, label='root alone (deposit / 3.68 eV)')
ax.axvline(MPV, color='C3', lw=2, label=f'most probable {MPV / 1e3:.1f} ke')
ax.axvline(qn.mean(), color='C3', ls='--', label=f'mean {qn.mean() / 1e3:.1f} ke')
ax.axvline(mip_mpv_e, color='k', ls=':', label=f'textbook most probable {mip_mpv_e / 1e3:.1f} ke')
ax.axvline(mip_mean_e, color='k', ls='-.', label=f'textbook mean {mip_mean_e / 1e3:.1f} ke')
ax.axvline(THRESHOLD, color='0.5', lw=2, label=f'counting threshold {THRESHOLD / 1e3:g} ke')
ax.set_xlim(0, 40000)
ax.set_xlabel('charge [e]')
ax.set_ylabel('normal crossings per 500 e')
ax.set_yscale('log')
ax.set_ylim(0.5, None)
ax.legend(fontsize=8)
ax = axs[1]
dq = GR['group_q'] - GR['group_edep_keV'] * 1e3 / EV_PER_PAIR
ax.hist(dq[NORMAL_G], bins=np.arange(-15.5, 16), histtype='stepfilled', color='C0', alpha=0.6, label='normal crossings')
ax.hist(dq, bins=np.arange(-15.5, 16), histtype='step', color='k', lw=1.2, label=f'all {NG:,} groups')
ax.set_xlabel(f'group_q - group_edep_keV / {EV_PER_PAIR} eV [e]')
ax.set_ylabel('groups')
ax.legend(fontsize=9)
fig.suptitle(f'shower {EVENT}: {int(NORMAL_G.sum()):,} normal crossings')
plt.show()
rel_dq = np.abs(dq[NORMAL_G]) / (GR['group_edep_keV'][NORMAL_G] * 1e3 / EV_PER_PAIR)
print(f"charge - energy / {EV_PER_PAIR} eV: standard deviation {dq[NORMAL_G].std():.2f} e for normal crossings, at most "
      f"{np.abs(dq[NORMAL_G]).max():.1f} e ({100 * rel_dq.max():.3f}% of the charge); all groups at most {np.abs(dq).max():.1f} e")

The charge is the deposited energy divided by 3.68 eV, to a few electrons (right). The left plot
has a logarithmic scale to show the tail. The root alone gives a narrow peak. Its delta rays move
the group's peak up a little and add a long tail to high values: the Landau tail of a thin layer
(the printed medians and quantiles). The next cell measures where the tail comes from, and why
the measured mean lies below the textbook's.

In [ ]:
first_gen = joins & (up == root_node)                       # delta rays joined straight to their root
ke_kids = np.bincount(group_of_node, np.where(first_gen, (ST['E'][node_part] - ST['mass'][node_part]) * 1e9, 0), NG)
lost_by_root = node_e[rn_] + ke_kids                        # its own deposit + the kinetic energy it gave its delta rays
escaped = lost_by_root - g_dep                              # what the delta rays took out of the slab
tail = NORMAL_G & (qg > 2 * MPV)
delta_share = dshare                                        # share of the group's deposit made by its delta rays
print(f"tail: {int(tail.sum())} normal crossings ({100 * tail.sum() / NORMAL_G.sum():.1f}%) have more than twice the "
      f"most probable charge and hold {100 * qg[tail].sum() / qn.sum():.1f}% of the charge; in "
      f"{100 * np.mean(delta_share[tail] > 0.5):.0f}% of them the delta rays made more than half of it")
print(f"normal crossings without any delta ray: {100 * np.mean(GR['group_n_particles'][NORMAL_G] == 1):.0f}%")
print(f"per normal crossing, on average: the root deposits {node_e[rn_][NORMAL_G].mean() / 1e3:.1f} keV and gives "
      f"{ke_kids[NORMAL_G].mean() / 1e3:.1f} keV to delta rays, a loss of {lost_by_root[NORMAL_G].mean() / 1e3:.1f} keV "
      f"= {lost_by_root[NORMAL_G].mean() / 1e3 / mip_mean_keV:.2f} x the textbook mean of {mip_mean_keV:.1f} keV")
print(f"the delta rays take {escaped[NORMAL_G].mean() / 1e3:.1f} keV of it out of the slab, so the group deposits "
      f"{g_dep[NORMAL_G].mean() / 1e3:.1f} keV = {g_dep[NORMAL_G].mean() / EV_PER_PAIR:,.0f} e")

below = qg < THRESHOLD
print(f"\nbelow the counting threshold of {THRESHOLD:g} e: none of the normal crossings (smallest {qn.min():,.0f} e); of "
      f"all {NG:,} groups, {int(below.sum())} ({100 * below.mean():.2f}%) holding {100 * qg[below].sum() / qg.sum():.3f}% "
      f"of the charge")
print(f"those groups: roots {', '.join(f'{pdg_name(t)} {n}' for t, n in zip(*np.unique(GR['group_root_pdg'][below], return_counts=True)))}; "
      f"median path {np.median(GR['group_path_um'][below]):.2f} um, median root momentum "
      f"{np.median(GR['group_root_p_MeV'][below]) * 1e3:.0f} keV/c; roots that cross the slab: "
      f"{int(np.sum(below & (r_zmin < 1) & (r_zmax > THICK - 1)))}")
assert not np.any(qn < THRESHOLD), "a normal crossing below the threshold"

The most probable charge of a normal crossing lies close to the rough textbook value (the printed
ratio). The tail is made by delta rays: in the crossings above twice the most probable charge, the
delta rays make more than half of it. The mean lies below the textbook mean because delta rays
carry part of the energy out of the slab. The energy the root loses by ionisation (its own deposit
plus what it gives its delta rays) is a little above the textbook mean, but the group keeps only
what stays in the slab (the printed values).

Every normal crossing passes the counting threshold easily. The groups below it are slow, have
very short paths, and none crosses the slab; together they hold a negligible share of the charge.

## 7. The footprint of one crossing on the pixels

How many pixels does a crossing light, and how is its charge shared among them? Two things set
this: the track's slant, and the sideways spread of the charge while it drifts (diffusion).

**Slant.** For groups whose root crosses the slab, we bin by the root's path over the thickness
(near 1 for a normal crossing, larger for a slanted one) and measure the pixels that hold charge,
the pixels above 1000 e, and the share of the group's charge in its brightest pixel.

In [ ]:
o = np.lexsort((-pix_q, pix_g))
first_row = o[np.r_[True, np.diff(pix_g[o]) != 0]]                 # the brightest pixel of every group
q_first = CH['group_q'][pix_g[first_row]]
brightest = np.zeros(n_groups)
brightest[pix_g[first_row]] = np.divide(pix_q[first_row], q_first, out=np.zeros(len(first_row)), where=q_first > 0)
n_above = np.bincount(pix_g, pix_q > THRESHOLD, n_groups)
n_pix = CH['group_n_pix']
loc = slice(G0, G0 + NG)                                          # the groups of shower 2 in the batch arrays
through = (r_zmin < 1) & (r_zmax > THICK - 1)
slant = r_path / THICK
EDGES = [0.0, NORMAL, 1.2, 1.5, 2.0, 3.0, 5.0, np.inf]
print(f"{'path / thickness':18s} {'groups':>7s} {'pixels with charge':>19s} {'above 1000 e':>13s} "
      f"{'brightest pixel share':>22s}")
print(f"{'':18s} {'':>7s} {'median    mean':>19s} {'median  mean':>13s} {'median':>22s}")
SLANT = []
for a, b in zip(EDGES[:-1], EDGES[1:]):
    m = through & (slant >= a) & (slant < b)
    if not m.any():
        continue
    SLANT.append((a, b, int(m.sum()), np.median(n_pix[loc][m]), n_pix[loc][m].mean(), np.median(n_above[loc][m]),
                  n_above[loc][m].mean(), np.median(brightest[loc][m])))
    lab = (f"up to {b:.2f}" if a == 0 else f"{a:.2f} to {b:.2f}") if np.isfinite(b) else f"above {a:.2f}"
    print(f"{lab:18s} {m.sum():7,d} {SLANT[-1][3]:8.0f} {SLANT[-1][4]:8.1f} {SLANT[-1][5]:7.0f} {SLANT[-1][6]:6.1f} "
          f"{SLANT[-1][7]:22.3f}")
alone = NORMAL_G & (GR['group_n_particles'] == 1)
withd = NORMAL_G & (GR['group_n_particles'] > 1)
print(f"\nnormal crossings without a delta ray: {int(alone.sum())}, {n_above[loc][alone].mean():.2f} pixels above "
      f"{THRESHOLD:g} e on average; with delta rays: {int(withd.sum())}, {n_above[loc][withd].mean():.2f}")

A normal crossing puts nearly all its charge into one pixel. As the track slants, it crosses more
pixels and the charge spreads over them. The pixels "with charge" include faint ones, below
1000 e, which is why there are more of them than pixels above 1000 e.

**Diffusion.** Take tracks so straight that they hit one point of the plane: groups whose root
and delta rays together stay within a couple of um sideways (from `group_u_min_um` to
`group_u_max_um` and the same in v; the cut is printed), over the whole thickness, in one tile;
all 20 showers of the batch. For each, we find the pixel edge nearest to its energy-weighted
centre, call the centre's position from that edge `x`, and measure the share of its charge on the
side `x > 0` of the edge, as a function of `x`. Without diffusion the share would jump from none
to all at the edge. A Gaussian spread of width `sigma`, centred `x0` from the track,
gives `Phi((x - x0) / sigma)`, the normal cumulative distribution; we fit it.

In [ ]:
LAT_CUT = 2.0                                                      # um
lat = np.hypot(CH['group_u_max_um'] - CH['group_u_min_um'], CH['group_v_max_um'] - CH['group_v_min_um'])
STRAIGHT = ((CH['group_w_min_um'] < 1) & (CH['group_w_max_um'] > THICK - 1) & (CH['group_n_tiles'] == 1) &
            (lat <= LAT_CUT))
print(f"straight tracks: the group's sideways extent (diagonal of its u, v box) at most {LAT_CUT:g} um")
qtot = np.maximum(CH['group_q'], 1e-9)
SCURVE = {}
for name, pitch, cen, ip in (('v', PITCH_V, CH['group_v_centroid_um'], pix_iv), ('u', PITCH_U, CH['group_u_centroid_um'], pix_iu)):
    edge = np.round(cen / pitch).astype(np.int64)                  # the nearest pixel edge, in pixels
    x = cen - edge * pitch                                         # the centroid's distance to it [um]
    far = np.bincount(pix_g, pix_q * (ip >= edge[pix_g]), n_groups) / qtot
    m = STRAIGHT & (np.abs(x) < 8)
    (sig, x0), cov = curve_fit(lambda t, s, c: norm.cdf((t - c) / s), x[m], far[m], p0=[2.0, 0.0])
    SCURVE[name] = dict(x=x[m], far=far[m], sigma=sig, x0=x0, err=np.sqrt(np.diag(cov)), n=int(m.sum()))
    print(f"along {name}: {int(m.sum()):,} straight tracks within 8 um of a pixel edge; sigma {sig:.2f} +- "
          f"{np.sqrt(cov[0, 0]):.2f} um, x0 {x0:+.3f} +- {np.sqrt(cov[1, 1]):.3f} um")
print(f"straight tracks in all: {int(STRAIGHT.sum()):,}; their brightest pixel holds a median share of "
      f"{np.median(brightest[STRAIGHT]):.4f}, and more than 90% in {100 * np.mean(brightest[STRAIGHT] > 0.9):.1f}% of them")
print(f"pixels above {THRESHOLD:g} e per straight track: mean {n_above[STRAIGHT].mean():.2f}; two or more for "
      f"{100 * np.mean(n_above[STRAIGHT] >= 2):.1f}% of them")
for name in ('v', 'u'):
    assert abs(SCURVE[name]['x0']) < 0.1, f"the charge lands {SCURVE[name]['x0']:.2f} um beside the track along {name}"
print("check: along v and u the charge lands within 0.1 um of the track (no sideways drift, as without a magnetic field)")

dist = np.minimum.reduce([CH['group_v_centroid_um'] % PITCH_V, PITCH_V - CH['group_v_centroid_um'] % PITCH_V,
                          CH['group_u_centroid_um'] % PITCH_U, PITCH_U - CH['group_u_centroid_um'] % PITCH_U])
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
ax = axs[0]
xb = np.linspace(-8, 8, 33)
xc = 0.5 * (xb[1:] + xb[:-1])
for (name, s), col in zip(SCURVE.items(), ('C0', 'C3')):
    idx = np.digitize(s['x'], xb) - 1
    mean = np.array([s['far'][idx == k].mean() if np.any(idx == k) else np.nan for k in range(len(xc))])
    ax.plot(xc, mean, 'o', color=col, ms=4, label=f'along {name}: {s["n"]:,} tracks')
    xf = np.linspace(-8, 8, 161)
    ax.plot(xf, norm.cdf((xf - s['x0']) / s['sigma']), '-', color=col, lw=1,
            label=f'fit: sigma {s["sigma"]:.2f} um, x0 {s["x0"]:+.2f} um')
ax.axvline(0, color='0.6', lw=0.8)
ax.set_xlabel('x: track position from the nearest pixel edge [um]')
ax.set_ylabel('share of the charge on the side x > 0')
ax.legend(fontsize=8, loc='upper left')
ax = axs[1]
db = np.linspace(0, PITCH_U / 2, 26)
dc = 0.5 * (db[1:] + db[:-1])
idx = np.digitize(dist[STRAIGHT], db) - 1
b_mean = [np.mean(brightest[STRAIGHT][idx == k]) if np.any(idx == k) else np.nan for k in range(len(dc))]
ax.plot(dc, b_mean, 'o-', color='C2', ms=4)
ax.set_xlabel("distance from the track to the nearest pixel edge [um]")
ax.set_ylabel('mean share of the charge in the brightest pixel')
ax.set_ylim(0.45, 1.02)
fig.suptitle(f'straight tracks of the {int(CH["num_events"])} showers: how the charge is shared at a pixel edge')
plt.show()

Seen at a pixel edge, the charge spreads sideways by little more than a micrometre (the fitted
`sigma`, one number for a track that starts charge at every depth of the slab), the same along
both pixel directions, and lands where the track is (`x0` is zero within its error). So a
straight track shares its charge with a neighbour only when it passes within a few `sigma` of an
edge; otherwise one pixel holds nearly all of it. Diffusion alone therefore puts a second pixel
above 1000 e only for the straight tracks that pass close to an edge (the printed share). The slant
spreads the charge far more (the table above), and delta rays add a little (the printed means).

**A gallery.** Six groups of shower 2, chosen by rule (printed), drawn as pixel maps in electrons
with the same colour scale. The dots are the midpoints of the group's steps: black for the root,
red for its delta rays. Each title gives the root's type and its momentum at its first step in
the plane, its path, the group's charge, and its pixels with charge (with those above 1000 e in
brackets).

In [ ]:
row0 = np.searchsorted(pix_g, np.arange(n_groups))
row1 = np.searchsorted(pix_g, np.arange(n_groups), side='right')
lat_v = GR['group_v_max_um'] - GR['group_v_min_um']
lat_u = GR['group_u_max_um'] - GR['group_u_min_um']
GAL_NV, GAL_NU = 11, 17                                      # pixels per panel
fits = (lat_v < (GAL_NV - 2) * PITCH_V) & (lat_u < (GAL_NU - 2) * PITCH_U)   # small enough for a panel
end_w = ST['vex'][node_part[rn_]] * np.cos(phi_node[rn_]) + ST['vey'][node_part[rn_]] * np.sin(phi_node[rn_])
stops = np.abs(end_w - sc.si_layer_centers()[node_lay[rn_]]) <= THICK / 2e3       # the root ends inside its slab
bright2 = brightest[loc]
q2 = GR['group_q']
npart2 = GR['group_n_particles']
fast = GR['group_root_p_MeV'] > 5.0                          # MeV/c: a track that stays straight
diag = (lat_v / PITCH_V > 0.5 * lat_u / PITCH_U) & (lat_v / PITCH_V < 2 * lat_u / PITCH_U)
taken = np.zeros(NG, bool)


def pick(mask, key):
    # the group of the mask, not shown yet, with the largest key
    idx = np.flatnonzero(mask & ~taken)
    g = int(idx[np.argmax(key[idx])])
    taken[g] = True
    return g


def n_of(n, word):
    return f"{n} {word}" + ("" if n == 1 else "s")


GALLERY = [
    ('one pixel', pick(NORMAL_G & (npart2 == 1) & (bright2 > 0.99), -np.abs(q2 - MPV))),
    ('a line along u', pick(through & one_tile & fits & fast & (npart2 <= 2) & (lat_u > 3 * lat_v), lat_u)),
    ('a line along v', pick(through & one_tile & fits & fast & (npart2 <= 2) & (lat_v > 3 * lat_u), lat_v)),
    ('a delta-ray blob', pick(NORMAL_G & fits & (dshare > 0.5) & (np.hypot(lat_u, lat_v) < 150),
                              n_above[loc] * 1e7 + g_dep)),
    ('a steep track', pick(through & one_tile & fits & fast & diag & (npart2 <= 3), r_path)),
    ('a stopping particle', pick(one_tile & fits & stops & ~through & (np.abs(GR['group_root_pdg']) == 11), q2)),
]
print(f"panels of {GAL_NV} x {GAL_NU} pixels ({GAL_NV * PITCH_V:g} x {GAL_NU * PITCH_U:g} um); rules, all in one tile "
      "and small enough for a panel:")
print("  one pixel: a normal crossing without delta rays, over 99% of its charge in one pixel, the charge nearest the "
      "most probable")
print("  a line along u (v): a root that crosses the slab, above 5 MeV/c, at most one delta ray, more than 3 times "
      "longer along u (v) than along v (u); the longest")
print("  a delta-ray blob: a normal crossing whose delta rays make over half of its deposit, within 150 um sideways; "
      "the most pixels above 1000 e, then the largest deposit")
print("  a steep track: a root that crosses the slab, above 5 MeV/c, at most two delta rays, as many pixels along v as "
      "along u within a factor 2; the longest path")
print("  a stopping particle: an electron or positron that ends in the slab without crossing it; the largest charge\n")
for name, g in GALLERY:
    print(f"{name:20s}: group {G0 + g:,}, {pdg_name(GR['group_root_pdg'][g])} of {GR['group_root_p_MeV'][g]:.2f} MeV/c, "
          f"root path {r_path[g]:.0f} um, {n_of(int(npart2[g]), 'particle')}, {q2[g]:,.0f} e = {q2[g] / MPV:.1f} x the "
          f"most probable charge, on {n_of(int(GR['group_n_pix'][g]), 'pixel')}")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
gnorm = LogNorm(vmin=1, vmax=max(pix_q[row0[G0 + g]:row1[G0 + g]].max() for _, g in GALLERY))
for ax, (name, g) in zip(axs.ravel(), GALLERY):
    r = slice(row0[G0 + g], row1[G0 + g])
    gv, gu, gq = pix_iv[r], pix_iu[r], pix_q[r]
    k = np.flatnonzero(group_of_step == g)
    iv0 = int(gv.min() + gv.max() + 1) // 2 - GAL_NV // 2              # the panel centred on the group's pixels
    iu0 = int(gu.min() + gu.max() + 1) // 2 - GAL_NU // 2
    img = np.zeros((GAL_NU, GAL_NV))
    m = (gv >= iv0) & (gv < iv0 + GAL_NV) & (gu >= iu0) & (gu < iu0 + GAL_NU)
    img[gu[m] - iu0, gv[m] - iv0] = gq[m]
    ext = (iv0 * PITCH_V, (iv0 + GAL_NV) * PITCH_V, iu0 * PITCH_U, (iu0 + GAL_NU) * PITCH_U)
    im = ax.imshow(np.where(img >= 1, img, np.nan), origin='lower', interpolation='nearest', extent=ext, norm=gnorm,
                   cmap='viridis')
    for jy, jx in zip(*np.nonzero(img > THRESHOLD)):
        ax.add_patch(Rectangle((ext[0] + jx * PITCH_V, ext[2] + jy * PITCH_U), PITCH_V, PITCH_U, fill=False, ec='k', lw=0.8))
    rs = tmc[k] == node_part[rn_[g]]
    ax.plot(SEG['v'][k][rs], SEG['u'][k][rs], '.', color='k', ms=3)
    ax.plot(SEG['v'][k][~rs], SEG['u'][k][~rs], '.', color='C3', ms=3)
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    ax.set_xticks(ext[0] + PITCH_V * np.arange(0, GAL_NV + 1, 2))
    ax.set_yticks(ext[2] + PITCH_U * np.arange(0, GAL_NU + 1, 4))
    ax.set_xlabel('v [um]', fontsize=9)
    ax.set_ylabel('u [um]', fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_title(f"{name}: {pdg_name(GR['group_root_pdg'][g])} {GR['group_root_p_MeV'][g]:.2f} MeV/c, path {r_path[g]:.0f} um\n"
                 f"{q2[g] / 1e3:.1f} ke, {n_of(int(GR['group_n_pix'][g]), 'pixel')} ({int(n_above[G0 + g])} above 1 ke)",
                 fontsize=10)
    assert m.all(), f"{name}: {int(np.sum(~m))} of its pixels lie outside the panel"
print(f"check: every pixel of the six groups lies inside its panel")
fig.colorbar(im, ax=axs, label='charge per pixel [e]', shrink=0.6)
fig.suptitle(f'six groups of shower {EVENT}, {GAL_NV} x {GAL_NU} pixels each; black outlines: pixels above '
             f'{THRESHOLD:g} e; dots: step midpoints (black root, red delta rays)')
plt.show()

The charge sits under the steps. A normal crossing makes one bright pixel. A slanted track lights
a line of pixels. Delta rays that travel sideways widen the footprint around the track. A slow
electron that stops in the silicon wanders and leaves several times the most probable charge of
a crossing (the printed ratios).

## 8. The plane

Notebook 10 starts from the charge image of one sensor plane: shower 2, the module that faces the
beam, and the layer where the shower leaves the most charge there. We make that image here by
adding the pixel charges of the plane's groups on the plane's pixel grid. `reticles.py` cuts
images like this one into reticles.

In [ ]:
on_axis = GR['group_module'] == AXIS_MODULE
q_layer = np.bincount(GR['group_layer'][on_axis], q2[on_axis], N_LAYERS)
LAYER = int(np.argmax(q_layer))
plane_groups = np.flatnonzero(on_axis & (GR['group_layer'] == LAYER)) + G0
rows = np.concatenate([np.arange(row0[g], row1[g]) for g in plane_groups])
iv_lo, iu_lo = int(pix_iv[rows].min()), int(pix_iu[rows].min())
nv, nu = int(pix_iv[rows].max()) - iv_lo + 1, int(pix_iu[rows].max()) - iu_lo + 1
IMAGE = np.bincount((pix_iu[rows] - iu_lo) * nv + (pix_iv[rows] - iv_lo), pix_q[rows], nu * nv).reshape(nu, nv)
q_plane = float(CH['group_q'][plane_groups].sum())
rel = abs(IMAGE.sum() - q_plane) / q_plane
assert rel < 1e-6, f"the image holds {IMAGE.sum():.1f} e, its groups {q_plane:.1f} e"
print(f"plane: shower {EVENT}, module {AXIS_MODULE}, layer {LAYER} ({q_layer[LAYER] / 1e6:.2f} million e, the most of "
      f"any layer); {len(plane_groups)} groups, {len(rows):,} (group, pixel) rows")
print(f"image: {nv:,} x {nu:,} pixels (v x u), {np.count_nonzero(IMAGE):,} of them with charge; "
      f"largest pixel {IMAGE.max():,.0f} e")
print(f"check: the image holds {IMAGE.sum():,.1f} e, the plane's groups {q_plane:,.1f} e (relative difference {rel:.1e})")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True, gridspec_kw={'width_ratios': [1, 1.5]})
ax = axs[0]
CELL = 0.2                                                             # mm
v_c = (pix_iv[rows] + 0.5) * PITCH_V / 1e3
u_c = (pix_iu[rows] + 0.5) * PITCH_U / 1e3
edges = np.arange(-10, 10 + CELL / 2, CELL)
H, _, _ = np.histogram2d(v_c, u_c, bins=[edges, edges], weights=pix_q[rows])
m = ax.pcolormesh(edges, edges, np.where(H > 0, H, np.nan).T, norm=LogNorm(vmin=1e2, vmax=H.max()), cmap='viridis')
fig.colorbar(m, ax=ax, label=f'charge per {CELL:g} x {CELL:g} mm cell [e]', shrink=0.8)
ZV, ZU = 32, 48                                                        # the zoom: 32 x 48 pixels around the axis
ax.add_patch(Rectangle((-ZV / 2 * PITCH_V / 1e3, -ZU / 2 * PITCH_U / 1e3), ZV * PITCH_V / 1e3, ZU * PITCH_U / 1e3,
                       fill=False, ec='red', lw=1.2))
ax.set_aspect('equal')
ax.set_xlabel('v [mm]')
ax.set_ylabel('u [mm]')
ax.set_title(f'the central 20 x 20 mm (red: the zoom)', fontsize=10)
ax = axs[1]
zv0, zu0 = -ZV // 2 - iv_lo, -ZU // 2 - iu_lo                        # the zoom in image indices
assert 0 <= zv0 and zv0 + ZV <= nv and 0 <= zu0 and zu0 + ZU <= nu, "the zoom leaves the image"
zoom = IMAGE[zu0:zu0 + ZU, zv0:zv0 + ZV]
ext = (-ZV / 2 * PITCH_V, ZV / 2 * PITCH_V, -ZU / 2 * PITCH_U, ZU / 2 * PITCH_U)
im = ax.imshow(np.where(zoom >= 1, zoom, np.nan), origin='lower', interpolation='nearest', extent=ext,
               norm=LogNorm(vmin=1, vmax=zoom.max()), cmap='viridis')
fig.colorbar(im, ax=ax, label='charge per pixel [e]', shrink=0.8)
ax.plot(0, 0, '+', color='red', ms=14, mew=2)
ax.set_xlabel('v [um]  (50 um pixels)')
ax.set_ylabel('u [um]  (25 um pixels)')
ax.set_title(f'the core, pixel by pixel ({ZV} x {ZU} pixels; red cross: the shower axis)', fontsize=10)
fig.suptitle(f'the charge image of shower {EVENT}, module {AXIS_MODULE}, layer {LAYER}')
plt.show()
r_mm = np.hypot(v_c, u_c)
print(f"share of the plane's charge within 1 mm of the axis: {pix_q[rows][r_mm < 1].sum() / q_plane:.2f}; in the zoom: "
      f"{zoom.sum() / q_plane:.2f}; outside the left picture: {pix_q[rows][(np.abs(v_c) >= 10) | (np.abs(u_c) >= 10)].sum() / q_plane:.3f}")

In [ ]:
in_shower = (pix_g >= G0) & (pix_g < G0 + NG)
q_shower_pix = float(pix_q[in_shower].sum())
q_shower = float(q2.sum())
pairs_shower = E_TOTAL_KEV * 1e3 / EV_PER_PAIR
assert abs(q_shower_pix - q_shower) <= 1e-6 * q_shower, "the shower's pixels do not add up to its groups"
assert abs(q_shower - pairs_shower) <= np.sum(RB['job_n_seg']), "the shower's charge is not its energy / 3.68 eV"
print(f"all planes of shower {EVENT}: pixels {q_shower_pix:,.0f} e, groups {q_shower:,.0f} e; deposited energy / "
      f"{EV_PER_PAIR} eV = {pairs_shower:,.0f} pairs; charge / pairs = {q_shower / pairs_shower:.7f}")
print(f"the plane above holds {100 * q_plane / q_shower:.1f}% of the shower's charge")

The image adds up to the charge of the plane's groups, and the sum over all planes is the
shower's deposited energy divided by 3.68 eV. Notebook 10 takes it from here: it reads this image
as the detector would (pixels on or off), cuts it into reticles, and defines what a reticle should
count.

## 9. Summary

The points below are printed from the numbers measured in this notebook.

In [ ]:
print(f"1. Shower {EVENT} of batch 0 ({float(JB['sim_gun_energy_GeV']):g} GeV {JB['sim_gun_particle']}) leaves "
      f"{E_TOTAL_KEV / 1e3:,.1f} MeV in the silicon in {n_step:,} steps of {n_part:,} particles; "
      f"{100 * at_limit.mean():.0f}% of the steps stop at the {STEP_LIMIT:g} um limit.")
print(f"2. The steps form {N_NODES:,} nodes (a particle in a plane) and {NG:,} groups; delta rays and their own delta "
      f"rays make {100 * nonroot_share:.0f}% of the deposit. Our loop over the grouping rule, and deposit_jobs.py run "
      f"again, reproduce every group and job of the batch for this shower.")
print(f"3. The groups become {NJ:,} jobs from {len(st):,} pieces of at most {PIECE:g} um; the pieces hold the steps' "
      f"energy (difference {diff:+.1e} eV) and every piece lies within its tile plus half a piece.")
print(f"4. PIXELAV makes the pairs as its rule says (all {len(made):,} jobs of the batch within one pair per piece; "
      f"for the {NJ:,} jobs of shower {EVENT} the extra pairs scatter with {zs.std():.2f} of their expected spread), "
      f"none outside a window, and collects {ratio_all:.9f} of the pairs transported.")
print(f"5. A normal crossing gives most probably {MPV:,.0f} e (textbook about {mip_mpv_e:,.0f} e) and on average "
      f"{qn.mean():,.0f} e (textbook {mip_mean_e:,.0f} e); delta rays make the tail and carry "
      f"{escaped[NORMAL_G].mean() / 1e3:.1f} keV per crossing out of the slab. None falls below {THRESHOLD:g} e.")
print(f"6. The charge spreads sideways by {SCURVE['v']['sigma']:.1f} um and lands under the track; a normal crossing "
      f"puts a median {SLANT[0][7]:.3f} of its charge in one pixel, a slanted or curling track lights more.")
print(f"7. The charge image of the plane (module {AXIS_MODULE}, layer {LAYER}) holds {q_plane:,.0f} e; all planes "
      f"together hold {q_shower:,.0f} e = {q_shower / pairs_shower:.6f} x the deposited energy / {EV_PER_PAIR} eV.")
print("\nNext, notebook 10: this plane as the detector sees it, cut into reticles.")
peak = None
try:
    import resource
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1e6 if sys.platform == 'darwin' else 1e3)
except ImportError:
    pass
print(f"\nrun time {time.time() - T_START:.0f} s" + (f", peak memory {peak:,.0f} MB" if peak else ""))